In [ ]:
import ee

ee.Authenticate()

In [ ]:
ee.Initialize(project='ee-cmber61')

In [ ]:
import geopandas as gpd
import os


# Cargar shapefile de municipios
municipios_gdf = gpd.read_file("../data/Municipios_Diciembre_2025_shp/Munpio.shp")

# Filtrar solo los municipios que necesitas
codigos_interes = [
    '05034', '05042', '05093', '05107', '05113', '05120', '05154',
    '05361', '05411', '05501', '05543', '05579', '05628', '05647',
    '05652', '05819', '05854', '05887', '17380', '17444', '17495',
    '17614', '17662', '17867', '18592', '20238', '47170', '50350',
    '50711', '54206', '54250', '54720', '54810', '66456', '66572',
    '70001', '70429', '70713', '70820'
]

# Verificar columna de código — puede llamarse MPIO_CDPMP, COD_MPIO, etc.
print(municipios_gdf.columns.tolist())
municipios_gdf.head(2)

In [ ]:
municipios_filtrados = municipios_gdf[
    municipios_gdf['MpCodigo'].isin(codigos_interes)
].copy()

print(f"Municipios encontrados: {len(municipios_filtrados)}")

# Asegurar CRS en WGS84 para GEE
municipios_filtrados = municipios_filtrados.to_crs("EPSG:4326")

#os.makedirs("../data/landsat", exist_ok=True)


In [ ]:
import ee

# Re-autenticar con scopes completos
ee.Authenticate(force=True, scopes=[
    'https://www.googleapis.com/auth/earthengine',
    'https://www.googleapis.com/auth/drive'
])

ee.Initialize(project='ee-cmber61')

# Verificar que funciona
print(ee.String('Autenticación exitosa').getInfo())

In [ ]:
print(f"Filas en municipios_filtrados: {len(municipios_filtrados)}")
print(f"Códigos únicos: {municipios_filtrados['MpCodigo'].nunique()}")
print(municipios_filtrados['MpCodigo'].value_counts().head(10))

In [ ]:
import ee
import time


# Municipios con período diferente
municipios_reprocesar = {
    '05543': ('1995-01-01', '2000-12-31'),  # Peñol
    '05628': ('1995-01-01', '2000-12-31'),  # San Carlos
    '05361': ('1999-01-01', '2003-12-31'),  # Ituango
}

def get_ndvi(geometry_ee, inicio, fin):
    return (
        ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
        .filterDate(inicio, fin)
        .filterBounds(geometry_ee)
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
        .map(lambda img: img.normalizedDifference(['SR_B4', 'SR_B3'])
                           .rename('NDVI'))
        .median()
        .clip(geometry_ee)
    )

for codigo, (inicio, fin) in municipios_reprocesar.items():
    mun = municipios_filtrados[
        municipios_filtrados['MpCodigo'].astype(str).str.zfill(5) == codigo
    ].iloc[0]
    geom_ee = ee.Geometry(mun['geometry'].__geo_interface__)
    ndvi = get_ndvi(geom_ee, inicio, fin)
    
    task = ee.batch.Export.image.toDrive(
        image=ndvi,
        description=f'NDVI_{codigo}_v2',
        folder='landsat_ndvi',
        fileNamePrefix=f'ndvi_{codigo}_v2',
        region=geom_ee,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e10
    )
    task.start()
    print(f"✓ NDVI_{codigo} ({inicio[:4]}-{fin[:4]}) iniciado")
    time.sleep(1)

In [ ]:
# Función para calcular NDVI histórico por municipio
def get_ndvi_historico(geometry_ee, periodo_inicio, periodo_fin):
    """Calcula NDVI mediano para un período dado usando Landsat 5."""
    coleccion = (
        ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
        .filterDate(periodo_inicio, periodo_fin)
        .filterBounds(geometry_ee)
        .filter(ee.Filter.lt('CLOUD_COVER', 20))
        .map(lambda img: img.normalizedDifference(['SR_B4', 'SR_B3'])
                           .rename('NDVI')
                           .copyProperties(img, ['system:time_start']))
    )
    return coleccion.median().clip(geometry_ee)

In [ ]:
codigos_encontrados = set(municipios_filtrados['MpCodigo'].astype(str).str.zfill(5))
faltantes = set(codigos_interes) - codigos_encontrados
print(f"Municipio faltante: {faltantes}")

In [ ]:
import time

# Obtener tareas existentes para evitar duplicados
tareas_existentes = ee.batch.Task.list()
nombres_existentes = set(
    t.config['description'] 
    for t in tareas_existentes 
    if t.state in ['READY', 'RUNNING', 'COMPLETED']
)
print(f"Tareas ya existentes: {len(nombres_existentes)}")

errores = []
for _, row in municipios_filtrados.iterrows():
    codigo = str(row['MpCodigo']).zfill(5)
    nombre_tarea = f'NDVI_{codigo}'
    
    if nombre_tarea in nombres_existentes:
        print(f"  → {nombre_tarea} ya existe, saltando")
        continue
    
    print(f"\nProcesando: {codigo}")
    try:
        geom_ee = ee.Geometry(row['geometry'].__geo_interface__)
        ndvi = get_ndvi_historico(geom_ee, '1998-01-01', '2002-12-31')
        task = ee.batch.Export.image.toDrive(
            image=ndvi,
            description=nombre_tarea,
            folder='landsat_ndvi',
            fileNamePrefix=f'ndvi_{codigo}',
            region=geom_ee,
            scale=30,
            crs='EPSG:4326',
            maxPixels=1e10
        )
        task.start()
        print(f"  ✓ Tarea iniciada: {nombre_tarea}")
        time.sleep(1)
    except Exception as e:
        print(f"  ✗ Error: {e}")
        errores.append((codigo, str(e)))

print(f"\nNuevas tareas creadas: {39 - len(errores)}")
print(f"Errores: {len(errores)}")

In [ ]:
# Cancelar todas las tareas activas
tareas = ee.batch.Task.list()
canceladas = 0

for tarea in tareas:
    if tarea.state in ['READY', 'RUNNING']:
        tarea.cancel()
        canceladas += 1

print(f"Tareas canceladas: {canceladas}")

In [ ]:
tareas = ee.batch.Task.list()
activas = [t for t in tareas if t.state in ['READY', 'RUNNING']]
print(f"Tareas aún activas: {len(activas)}")

In [ ]:
import ee
import time

ee.Initialize(project='ee-cmber61')

# DEM SRTM desde GEE
dem = ee.Image('USGS/SRTMGL1_003').select('elevation')

tareas_existentes = ee.batch.Task.list()
nombres_existentes = set(
    t.config['description'] 
    for t in tareas_existentes 
    if t.state in ['READY', 'RUNNING', 'COMPLETED']
)

errores = []
for _, row in municipios_filtrados.iterrows():
    codigo = str(row['MpCodigo']).zfill(5)
    nombre_tarea = f'DEM_{codigo}'
    
    if nombre_tarea in nombres_existentes:
        print(f"  → {nombre_tarea} ya existe, saltando")
        continue

    try:
        geom_ee = ee.Geometry(row['geometry'].__geo_interface__)
        
        task = ee.batch.Export.image.toDrive(
            image=dem.clip(geom_ee),
            description=nombre_tarea,
            folder='dem_srtm',
            fileNamePrefix=f'dem_{codigo}',
            region=geom_ee,
            scale=30,
            crs='EPSG:4326',
            maxPixels=1e10
        )
        task.start()
        print(f"✓ {nombre_tarea} iniciado")
        time.sleep(1)

    except Exception as e:
        print(f"✗ Error {codigo}: {e}")
        errores.append((codigo, str(e)))

print(f"\nTareas DEM iniciadas. Errores: {len(errores)}")

In [ ]:
import ee
ee.Initialize(project='ee-cmber61')

tareas = ee.batch.Task.list()
ndvi_tareas = [t for t in tareas if t.config['description'].startswith('NDVI_')]

completadas = [t for t in ndvi_tareas if t.state == 'COMPLETED']
fallidas = [t for t in ndvi_tareas if t.state == 'FAILED']
corriendo = [t for t in ndvi_tareas if t.state in ['READY', 'RUNNING']]

print(f"Completadas: {len(completadas)}")
print(f"Fallidas: {len(fallidas)}")
print(f"Aún activas: {len(corriendo)}")

print(f"\nMunicipios con error:")
for t in fallidas:
    print(f"  {t.config['description']} — {t.status().get('error_message', 'sin mensaje')}")

In [ ]:
import time

# Solo los 4 municipios con error
municipios_error = {
    '66456': 'Marsella',
    '05093': 'Betulia', 
    '05034': 'Arboletes',
    '17614': 'Riosucio'
}

def get_ndvi_historico(geometry_ee, periodo_inicio, periodo_fin):
    coleccion = (
        ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
        .filterDate(periodo_inicio, periodo_fin)
        .filterBounds(geometry_ee)
        .filter(ee.Filter.lt('CLOUD_COVER', 50))  # subimos umbral a 50%
        .map(lambda img: img.normalizedDifference(['SR_B4', 'SR_B3'])
                           .rename('NDVI'))
    )
    return coleccion.median().clip(geometry_ee)

for codigo, nombre in municipios_error.items():
    print(f"\nProcesando: {codigo} - {nombre}")
    row = municipios_filtrados[
        municipios_filtrados['MpCodigo'].astype(str).str.zfill(5) == codigo
    ].iloc[0]
    
    geom_ee = ee.Geometry(row['geometry'].__geo_interface__)
    
    # Período más amplio: 1990-2005
    ndvi = get_ndvi_historico(geom_ee, '1990-01-01', '2005-12-31')
    
    # Verificar que tiene bandas antes de exportar
    try:
        info = ndvi.getInfo()
        print(f"  ✓ Imagen tiene bandas — exportando")
        task = ee.batch.Export.image.toDrive(
            image=ndvi,
            description=f'NDVI_{codigo}',
            folder='landsat_ndvi',
            fileNamePrefix=f'ndvi_{codigo}',
            region=geom_ee,
            scale=30,
            crs='EPSG:4326',
            maxPixels=1e10
        )
        task.start()
        print(f"  ✓ Tarea iniciada")
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    time.sleep(1)

In [ ]:
tareas = ee.batch.Task.list()
dem_tareas = [t for t in tareas if t.config['description'].startswith('DEM_')]

completadas = [t for t in dem_tareas if t.state == 'COMPLETED']
fallidas = [t for t in dem_tareas if t.state == 'FAILED']
corriendo = [t for t in dem_tareas if t.state in ['READY', 'RUNNING']]

print(f"Completadas: {len(completadas)}")
print(f"Fallidas: {len(fallidas)}")
print(f"Aún activas: {len(corriendo)}")

for t in fallidas:
    print(f"  {t.config['description']} — {t.status().get('error_message', 'sin mensaje')}")

In [ ]:
import time

pendientes = {
    '70713': 'San Onofre',
    '70820': 'Tolú Viejo'  
}

def get_ndvi_historico(geometry_ee, periodo_inicio, periodo_fin):
    coleccion = (
        ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
        .filterDate(periodo_inicio, periodo_fin)
        .filterBounds(geometry_ee)
        .filter(ee.Filter.lt('CLOUD_COVER', 50))
        .map(lambda img: img.normalizedDifference(['SR_B4', 'SR_B3'])
                           .rename('NDVI'))
    )
    return coleccion.median().clip(geometry_ee)

for codigo, nombre in pendientes.items():
    row = municipios_filtrados[
        municipios_filtrados['MpCodigo'].astype(str).str.zfill(5) == codigo
    ].iloc[0]
    
    geom_ee = ee.Geometry(row['geometry'].__geo_interface__)
    ndvi = get_ndvi_historico(geom_ee, '1990-01-01', '2005-12-31')
    
    task = ee.batch.Export.image.toDrive(
        image=ndvi,
        description=f'NDVI_{codigo}',
        folder='landsat_ndvi',
        fileNamePrefix=f'ndvi_{codigo}',
        region=geom_ee,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e10
    )
    task.start()
    print(f"✓ NDVI_{codigo} — {nombre} iniciado")
    time.sleep(1)